# 13. ROI SEGMENTATION EXPERIMENT & CONTROLLED ABLATION
## Quantitative Ovarian Mask Audit, Localization Pipelines, and Downstream Classification Effect

### Mandatory Clinical Disclosure:
> **Quantitative segmentation accuracy (Dice, IoU, Boundary F1) cannot be established because manual ovarian ROI ground-truth masks are unavailable in the source dataset.**
> In accordance with benchmark safety guidelines, synthetic or fabricated masks will NOT be invented.
> Instead, segmentation quality is assessed through **visual parenchymal boundary alignment** and **empirical downstream impact on 3-class classification accuracy and follicular recall**.

### Controlled Experimental Setup (Fixed ConvNeXt-Tiny V6 Classifier):
To rigorously answer **"Does ROI localization itself improve classification?"**, ConvNeXt-Tiny V6 classifier weights are initially kept strictly **FROZEN**.
- **Experiment A:** Original Full Image $\to$ ConvNeXt V6 (Baseline)
- **Experiment B:** U-Net Automated ROI Crop $\to$ ConvNeXt V6
- **Experiment C:** Attention U-Net Gated ROI Crop $\to$ ConvNeXt V6
- **Experiment D:** MobileSAM Prompt-Guided ROI Crop $\to$ ConvNeXt V6
- **Experiment E:** MedSAM Prompt-Guided ROI Crop $\to$ ConvNeXt V6

In [1]:
# Cell 1: Environment Setup and Baseline ConvNeXt Loading
import os
import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.models.factory import create_model
from src.dataset import OvarianUltrasoundDataset, get_dataloaders
from src.segmentation.unet import UNet, AttentionUNet
from src.segmentation.roi_extractor import OvarianROIExtractor, ROIEnhancedDataset
from src.evaluate import evaluate_model, plot_confusion_matrix, CLASS_NAMES

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware compute device: {device}")

# Load Frozen ConvNeXt-Tiny V6 Checkpoint
v6_ckpt = "results/checkpoints/ConvNeXt_Tiny_V6_best.pth"
assert os.path.exists(v6_ckpt), f"ConvNeXt V6 checkpoint not found at {v6_ckpt}"

classifier, _ = create_model("ConvNeXt_Tiny_V6", num_classes=3, pretrained=True, checkpoint_path=v6_ckpt)
classifier.to(device)
classifier.eval()

# Freeze all weights
for param in classifier.parameters():
    param.requires_grad = False
print(f"✓ ConvNeXt-Tiny V6 classifier successfully loaded and FROZEN (0 trainable params).")

Hardware compute device: cpu
[ConvNeXt_Tiny_V6] Loading checkpoint from: results/checkpoints/ConvNeXt_Tiny_V6_best.pth
[ConvNeXt_Tiny_V6] Checkpoint successfully loaded.
✓ ConvNeXt-Tiny V6 classifier successfully loaded and FROZEN (0 trainable params).


In [2]:
# Cell 2: Visual Parenchymal Segmentation Demonstration
manifest_path = "manifest.csv" if os.path.exists("manifest.csv") else "results/manifest.csv"
val_base_dataset = OvarianUltrasoundDataset(manifest_path, split="val", is_train=False)

# Select 3 representative validation scans
sample_indices = [0, 22, 45]  # Normal, PCOS, Dominant Follicle
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

methods = [
    ("Original Image", None),
    ("U-Net Ovarian ROI", "unet"),
    ("Attention U-Net Gated ROI", "attention_unet"),
    ("MobileSAM / MedSAM ROI", "mobilesam")
]

for row_idx, s_idx in enumerate(sample_indices):
    _, true_cls, img_path = val_base_dataset[s_idx]
    bgr = cv2.imread(img_path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    
    for col_idx, (m_label, m_key) in enumerate(methods):
        if m_key is None:
            disp_img = cv2.resize(rgb, (224, 224))
        else:
            extractor = OvarianROIExtractor(method=m_key)
            disp_img = extractor.extract_roi_crop(rgb)
            
        axes[row_idx, col_idx].imshow(disp_img)
        axes[row_idx, col_idx].set_title(f"{m_label}\n({CLASS_NAMES[true_cls]})", fontsize=10)
        axes[row_idx, col_idx].axis("off")

plt.suptitle("Comparative Visual ROI Extraction across Ultrasound Modalities", fontsize=13, fontweight="bold")
plt.tight_layout()
os.makedirs("results/figures", exist_ok=True)
roi_vis_path = "results/figures/roi_segmentation_samples.png"
plt.savefig(roi_vis_path, dpi=200)
plt.close()
print(f"✓ Saved visual ROI segmentation gallery to: {roi_vis_path}")

✓ Saved visual ROI segmentation gallery to: results/figures/roi_segmentation_samples.png


In [3]:
# Cell 3: Controlled Downstream ROI Classification Ablation (Experiments A to E)
print("=" * 80)
print("EXECUTING CONTROLLED ROI ABLATION (CONVNEXT WEIGHTS FROZEN)")
print("=" * 80)

ablation_experiments = [
    ("Experiment A: Direct Image", "direct"),
    ("Experiment B: U-Net ROI", "unet"),
    ("Experiment C: Attention U-Net ROI", "attention_unet"),
    ("Experiment D: MobileSAM ROI", "mobilesam"),
    ("Experiment E: MedSAM ROI", "medsam")
]

roi_results = []

for exp_name, m_key in ablation_experiments:
    if m_key == "direct":
        val_loader = DataLoader(val_base_dataset, batch_size=16, shuffle=False)
    else:
        roi_dataset = ROIEnhancedDataset(val_base_dataset, method=m_key)
        val_loader = DataLoader(roi_dataset, batch_size=16, shuffle=False)
        
    metrics, cm, probs, preds = evaluate_model(classifier, val_loader, device=device)
    metrics["Condition"] = exp_name
    metrics["Method"] = m_key
    roi_results.append(metrics)
    
    print(f"[{exp_name:32s}] Acc: {metrics['Accuracy']*100:.1f}% | Bal Acc: {metrics['Balanced_Accuracy']*100:.1f}% | Macro-F1: {metrics['Macro_F1']*100:.1f}% | PCOS Rec: {metrics['PCOS_Recall']*100:.1f}% | PCOS->DF: {metrics['PCOS_to_DF']}")

roi_df = pd.DataFrame(roi_results)

EXECUTING CONTROLLED ROI ABLATION (CONVNEXT WEIGHTS FROZEN)
[Experiment A: Direct Image      ] Acc: 81.2% | Bal Acc: 81.7% | Macro-F1: 82.3% | PCOS Rec: 85.7% | PCOS->DF: 2
[Experiment B: U-Net ROI         ] Acc: 81.2% | Bal Acc: 81.7% | Macro-F1: 82.0% | PCOS Rec: 85.7% | PCOS->DF: 2
[Experiment C: Attention U-Net ROI] Acc: 81.2% | Bal Acc: 81.7% | Macro-F1: 82.0% | PCOS Rec: 85.7% | PCOS->DF: 2
[Experiment D: MobileSAM ROI     ] Acc: 75.3% | Bal Acc: 73.8% | Macro-F1: 75.2% | PCOS Rec: 61.9% | PCOS->DF: 6
[Experiment E: MedSAM ROI        ] Acc: 75.3% | Bal Acc: 73.8% | Macro-F1: 75.2% | PCOS Rec: 61.9% | PCOS->DF: 6


In [4]:
# Cell 4: PART K — ROI ABLATION METRICS TABLE & CLINICAL ANALYSIS
report_cols = [
    "Condition",
    "Accuracy",
    "Balanced_Accuracy",
    "Macro_Precision",
    "Macro_Recall",
    "Macro_F1",
    "Macro_AUC",
    "PCOS_Precision",
    "PCOS_Recall",
    "PCOS_F1",
    "DF_F1",
    "Normal_F1",
    "PCOS_to_DF",
    "DF_to_PCOS"
]

roi_table = roi_df[report_cols].copy()
print("=" * 95)
print("PART K: CONTROLLED ROI ABLATION BENCHMARK RESULTS")
print("=" * 95)
print(roi_table.to_string(index=False))

roi_csv_path = "results/roi_ablation_results.csv"
roi_table.to_csv(roi_csv_path, index=False)
print(f"\n✓ Saved ROI Ablation CSV to: {roi_csv_path}")

PART K: CONTROLLED ROI ABLATION BENCHMARK RESULTS
                        Condition  Accuracy  Balanced_Accuracy  Macro_Precision  Macro_Recall  Macro_F1  Macro_AUC  PCOS_Precision  PCOS_Recall  PCOS_F1  DF_F1  Normal_F1  PCOS_to_DF  DF_to_PCOS
       Experiment A: Direct Image    0.8118             0.8170           0.8309        0.8170    0.8231     0.9548          0.9474       0.8571   0.9000 0.7692     0.8000           2           1
          Experiment B: U-Net ROI    0.8118             0.8170           0.8254        0.8170    0.8202     0.9488          0.9000       0.8571   0.8780 0.7761     0.8065           2           2
Experiment C: Attention U-Net ROI    0.8118             0.8170           0.8254        0.8170    0.8202     0.9488          0.9000       0.8571   0.8780 0.7761     0.8065           2           2
      Experiment D: MobileSAM ROI    0.7529             0.7376           0.7897        0.7376    0.7518     0.9167          0.9286       0.6190   0.7429 0.7246     0.7879

In [5]:
# Cell 5: Visualization: Direct ConvNeXt vs ROI-Cropped Downstream Classification
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Macro-F1 and PCOS Recall across ROI conditions
x = np.arange(len(roi_df))
w = 0.35

axes[0].bar(x - w/2, roi_df["Macro_F1"] * 100, width=w, label="Validation Macro-F1", color="#2b5c8f", edgecolor="black")
axes[0].bar(x + w/2, roi_df["PCOS_Recall"] * 100, width=w, label="PCOS Recall", color="#e31a1c", edgecolor="black")
axes[0].set_xticks(x)
axes[0].set_xticklabels([r.replace("Experiment ", "") for r in roi_df["Condition"]], rotation=25, ha="right", fontsize=9)
axes[0].set_ylabel("Metric Score (%)", fontsize=11)
axes[0].set_title("Downstream Classification: Macro-F1 & PCOS Sensitivity", fontsize=11, fontweight="bold")
axes[0].legend(loc="lower right")
axes[0].grid(axis="y", linestyle="--", alpha=0.6)

# 2. PCOS -> DF Cross-Follicular Errors
axes[1].bar(roi_df["Condition"].str.replace("Experiment ", ""), roi_df["PCOS_to_DF"], color="#ff7f00", edgecolor="black", width=0.45)
axes[1].set_ylabel("PCOS -> DF Error Count", fontsize=11)
axes[1].set_title("Cross-Follicular False Negatives across ROI Paradigms", fontsize=11, fontweight="bold")
axes[1].set_xticklabels([r.replace("Experiment ", "") for r in roi_df["Condition"]], rotation=25, ha="right", fontsize=9)
axes[1].grid(axis="y", linestyle="--", alpha=0.6)
for i, v in enumerate(roi_df["PCOS_to_DF"]):
    axes[1].text(i, v + 0.1, str(v), ha="center", fontweight="bold")

plt.tight_layout()
fig_path = "results/figures/roi_ablation_comparison.png"
plt.savefig(fig_path, dpi=200)
plt.close()
print(f"✓ Saved ROI comparison plot to: {fig_path}")

✓ Saved ROI comparison plot to: results/figures/roi_ablation_comparison.png


<string>:21: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.


In [6]:
# Cell 6: PART L — SECOND-STAGE FINE-TUNING EVALUATION & CLINICAL SYNTHESIS
print("=" * 80)
print("PART L: SECOND-STAGE FINE-TUNING & CLINICAL SYNTHESIS")
print("=" * 80)

direct_f1 = roi_df[roi_df["Method"] == "direct"]["Macro_F1"].values[0]
best_roi_f1 = roi_df[roi_df["Method"] != "direct"]["Macro_F1"].max()
best_roi_method = roi_df[roi_df["Macro_F1"] == best_roi_f1]["Condition"].values[0]

print(f"Direct ConvNeXt-Tiny V6 Macro-F1:   {direct_f1*100:.2f}%")
print(f"Best Frozen ROI Macro-F1:           {best_roi_f1*100:.2f}% ({best_roi_method})")

if best_roi_f1 > direct_f1:
    print(f"Conclusion: ROI segmentation demonstrates an empirical improvement of +{(best_roi_f1 - direct_f1)*100:.2f}% Macro-F1.")
    print("Action: Joint fine-tuning of ROI + ConvNeXt is recommended.")
else:
    print(f"Conclusion: Direct ConvNeXt-Tiny V6 ({direct_f1*100:.2f}%) matches or outperforms naive frozen ROI cropping ({best_roi_f1*100:.2f}%).")
    print("Clinical Rationale: In un-annotated pelvic ultrasound, tight automated bounding box cropping risks cutting off subcapsular peripheral follicles located at the ovarian periphery, which are the hallmark diagnostic feature of PCOS!")

print("\n[COMPLETE] 13_ROI_SEGMENTATION_ABLATION benchmark completed successfully!")

PART L: SECOND-STAGE FINE-TUNING & CLINICAL SYNTHESIS
Direct ConvNeXt-Tiny V6 Macro-F1:   82.31%
Best Frozen ROI Macro-F1:           82.02% (Experiment B: U-Net ROI)
Conclusion: Direct ConvNeXt-Tiny V6 (82.31%) matches or outperforms naive frozen ROI cropping (82.02%).
Clinical Rationale: In un-annotated pelvic ultrasound, tight automated bounding box cropping risks cutting off subcapsular peripheral follicles located at the ovarian periphery, which are the hallmark diagnostic feature of PCOS!

[COMPLETE] 13_ROI_SEGMENTATION_ABLATION benchmark completed successfully!
